# 13. Debt and stock-flow reconciliation

Reconcile modern Maastricht debt changes with B.9 and the stock-flow adjustment, showing that debt dynamics are not the mirror image of the annual balance.

**Reads**

- `outputs/tables/debt_stock_flow_reconciliation.csv`

**Writes**

- Nothing. The reconciliation table is persisted by the pipeline.

**Method reference:** `METHODOLOGY.md` section 14

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

%matplotlib inline

from portugal_fiscal_balance.analysis import figures

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. The reconciliation

$$\Delta Debt_t = -B_t + SFA_t.$$

The stock-flow adjustment absorbs everything that changes debt without passing
through the annual balance: financial-asset transactions, valuation effects,
timing differences and statistical adjustments. It is computed here as a residual
check on the persisted table.

In [ ]:
debt = pd.read_csv(TABLES / 'debt_stock_flow_reconciliation.csv')
debt_columns = [
    'year',
    'balance_m_eur',
    'debt_change_m_eur',
    'stock_flow_adjustment_m_eur',
    'debt_pct_gdp',
    'stock_flow_adjustment_pct_gdp',
    'reconciliation_error_m_eur',
]
general = debt.loc[debt['sector'].eq('general_government') & debt['year'].ge(2010), debt_columns]
display(general.round(3))
print('max |reconciliation residual| (M EUR):', float(debt['reconciliation_error_m_eur'].abs().max()))

In [ ]:
figure = figures.debt_and_stock_flow(debt)

## 2. What moves the debt ratio

The columns below split each annual debt change into minus the balance and the
stock-flow adjustment. In several years the adjustment is the larger term, which
is the point of running the reconciliation at all.

In [ ]:
figure = figures.debt_change_decomposition(debt)

In [ ]:
general_government = debt.loc[debt['sector'].eq('general_government')].copy()
general_government['sfa_share_abs_debt_change'] = (
    general_government['stock_flow_adjustment_m_eur'].abs()
    / general_government['debt_change_m_eur'].abs()
)
display(
    general_government.loc[
        general_government['sfa_share_abs_debt_change'].gt(1.0),
        ['year', 'balance_m_eur', 'debt_change_m_eur', 'stock_flow_adjustment_m_eur'],
    ].round(1)
)

## Interpretation limits

1. The stock-flow adjustment is a **residual category**, not a behavioural
   variable. A large value is a signal to consult the source documentation, not
   an anomaly by itself.
2. Reconciliation is available for the **modern regime only**, where debt and
   adjustment series are published on a consistent basis.
3. The debt ratio changes with **nominal GDP** as well as with debt, so a falling
   ratio does not imply falling debt.

---

[Previous: 12. Fixed-capital-formation diagnostic](12_investment_diagnostic.ipynb) | [Next: 14. Descriptive macroeconomic co-movement](14_macroeconomic_comovement.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```